# Search 'n Sort — S1B & S1C

Revision / content-lookup notebook for the searching and sorting algorithms in H2 Computing (9569).

## Contents
1. [Searching](#searching): linear, binary and hash-table search
2. [Sorting](#sorting): insertion, bubble, quicksort and mergesort
3. [Complexity lookup](#complexity-lookup)
4. [Test scaffold](#tests)

**How to use:** keep the logic and stopping conditions in memory; use the code as an exam-style implementation reference. Run the test scaffold after changing an algorithm.

In [9]:
#test list init - generates list of random numbers with number of terms n
import random
def reset_test_list():
    global test_list
    test_list = []
    n = 10
    lower_bound = 0
    upper_bound = n
    for i in range(n):
        test_list.append(random.randint(lower_bound, upper_bound))
    print(f"test list: {test_list}")
reset_test_list()

test list: [10, 3, 3, 5, 9, 6, 8, 8, 8, 3]


<a id="searching"></a>
## Searching

### Linear search

Probe everything in order along the line, 1 by 1 (hence linear). It works on **sorted or unsorted** data and returns the first matching index in this implementation.

**Stopping conditions:** return the index when found; return `-1` after the entire list has been searched.

- Best time: **O(1)** — found on the first term
- Average/worst time: **O(n)** — may sweep most/all of the list
- Extra space: **O(1)**

In [10]:
def LinearSearch(Arr, FindValue):
    for i in range(len(Arr)):
        if Arr[i] == FindValue:
            return i  # return index of first match
    return -1  # not found

### Binary search

Only works on a **sorted** list, but is way faster than linear search for large inputs. Take `FindValue`, compare it to the middle, then move to either the left or right side through a comparison with that middle value. This halves the searching scope for every pass until the target is found or the bounds cross.

**Recursive stopping conditions:**
- `Low > High` → target is absent, return `-1`
- middle value matches → return its index
- otherwise recurse with `middle - 1` or `middle + 1`, so the bounds always shrink

- Best time: **O(1)** — middle matches immediately
- Average/worst time: **O(log n)**
- Extra space: **O(1)** iterative; **O(log n)** recursive call stack

In [11]:
def BinarySearch(Arr, FindValue, Low, High):
    if Low > High:
        return -1
    middle = (Low + High) // 2
    if Arr[middle] == FindValue:
        return middle
    elif FindValue < Arr[middle]:
        return BinarySearch(Arr, FindValue, Low, middle - 1)
    else:
        return BinarySearch(Arr, FindValue, middle + 1, High)


### Hash-table search

Create a hash table and use a hash function to calculate the bucket/index to inspect. Collisions must be handled; the example below uses **separate chaining** (a list at each bucket).

- Average search time: **O(1)** when keys are spread well
- Worst search time: **O(n)** when many keys collide into one chain
- Extra space for one search: **O(1)**
- Storage for the whole hash table: **O(n)**

This is not the main exam-syntax version; for more comprehensive hash-table code, look at the ADT notebook. The important distinction is that O(1) is the **average**, not a guarantee.

In [12]:
#hashing with separate chaining (LLM written, in early 2025)
BUCKET_SIZE = 7
class Hash(object):
    def __init__(self, bucket):
        # Number of buckets
        self.__bucket = bucket
        # Hash table of size bucket
        self.__table = [[] for _ in range(bucket)]

    # hash function to map values to key
    def hashFunction(self, key):
        return (key % self.__bucket)

    def insertItem(self, key):
        # get the hash index of key
        index = self.hashFunction(key)
        self.__table[index].append(key)

    def deleteItem(self, key):
        # get the hash index of key
        index = self.hashFunction(key)

        # Check the key in the hash table
        if key not in self.__table[index]:
            return

        # delete the key from hash table
        self.__table[index].remove(key)

    # function to display hash table
    def displayHash(self):
        for i in range(self.__bucket):
            print("[%d]" % i, end='')
            for x in self.__table[i]:
                print(" --> %d" % x, end='')
            print()


# Drive Program
if __name__ == "__main__":
    # array that contains keys to be mapped
    reset_test_list()

    # Create a empty hash table of BUCKET_SIZE
    h = Hash(BUCKET_SIZE)

    # insert the keys into the hash table
    for x in test_list:
        h.insertItem(x)

    # delete 12 from the hash table
    h.deleteItem(x)
    # Display the hash table
    h.displayHash()

test list: [1, 3, 1, 2, 2, 9, 6, 1, 0, 4]
[0] --> 0
[1] --> 1 --> 1 --> 1
[2] --> 2 --> 2 --> 9
[3] --> 3
[4]
[5]
[6] --> 6


<a id="sorting"></a>
## Sorting

### Insertion sort

Front of list is the sorted portion and back is unsorted. Take the next object from the unsorted portion and compare it backwards through the sorted portion. Shift larger objects right until its insertion point is reached. This sweeps the list until the sorted portion is the size of the whole list.

**Useful invariant:** before each outer-loop pass, everything left of `i` is already sorted.

- Best time: **O(n)** — already sorted, so no shifting
- Average/worst time: **O(n²)**
- Extra space: **O(1)** (in-place)
- **Stable:** yes, because equal values are not shifted past each other

In [13]:
def InsertionSort(Arr):
    for i in range(1, len(Arr)):
        current = Arr[i]  # object taken from unsorted portion
        j = i - 1
        # shift larger sorted objects right until insertion spot found
        while j >= 0 and Arr[j] > current:
            Arr[j + 1] = Arr[j]
            j -= 1
        Arr[j + 1] = current  # insert into sorted portion
    return Arr

### Bubble sort

On every pass, look at the current object and next object. Swap to get them in the correct order, then move to the next index. Larger objects "bubble" towards the end. Keep passing until a full pass returns 0 swaps, which is when you know the whole list is sorted.

- Best time: **O(n)** with the `swapped` early-exit flag
- Average/worst time: **O(n²)**
- Extra space: **O(1)** (in-place)
- **Stable:** yes, when swapping only if left `>` right

Exam optimisation: after each pass, one more final element is fixed, so the inner range may be shortened. The current code is still correct without that optimisation.

In [14]:
def BubbleSort(Arr):
    n = len(Arr)
    swapped = True
    while swapped:
        swapped = False
        for i in range(n - 1):
            if Arr[i] > Arr[i + 1]:
                # swap to put current pair in correct order
                temp = Arr[i]
                Arr[i] = Arr[i + 1]
                Arr[i + 1] = temp
                swapped = True
        # after a no-swap pass, list is sorted
    return Arr

### Quicksort

A divide-and-conquer algorithm: recursively partition parts of the list to conquer each part bit by bit.

**Partition:** take the first value as `pivotValue`. Move `leftmark` until it finds a value larger than the pivot; move `rightmark` until it finds a value smaller than the pivot. Swap those misplaced objects and continue until the bounds cross. Finally swap the pivot into `rightmark`. The pivot is now in its **final sorted position**: values on its left are ≤ it and values on its right are ≥ it.

**Conquer:** recursively quicksort the left and right sub-arrays. Once a sub-array has zero or one object (`First >= Last`), it is already sorted and recursion stops. Every partition fixes another pivot, eventually sorting the entire list.

Quicksort is recursive, with this implementation processing left then right to sweep through everything.

- Best/average time: **O(n log n)** with reasonably balanced partitions
- Worst time: **O(n²)** with very unbalanced partitions (e.g. consistently poor pivot choices)
- Array storage: in-place; recursion stack **O(log n)** average, **O(n)** worst
- Usually **not stable**

In [15]:
def Partition(Arr, First, Last):
  pivotValue = Arr[First]
  leftmark = First + 1 #index of second left object
  rightmark = Last #index of right-most object
  done = False

  while not done:
    while leftmark <= rightmark and Arr[leftmark] <= pivotValue:
      #increment leftmark index until leftmark array value >= first term
      leftmark += 1
    while Arr[rightmark] >= pivotValue and rightmark >= leftmark:
      #decrease rightmark index until rightmark array value <= first term
      rightmark -= 1
    #the loops above will stop if rightmark approaches leftmark

    if rightmark < leftmark:
      #end the loop when rightmark index < leftmark index
      done = True
    else:
      #swap leftmark and rightmark value in array
      temp = Arr[leftmark]
      Arr[leftmark] = Arr[rightmark]
      Arr[rightmark] = temp
  #swap first term and rightmark value
  temp = Arr[First]
  Arr[First] = Arr[rightmark]
  Arr[rightmark] = temp
  return rightmark #output rightmark index

def Quicksort(Array, First, Last):
  if First < Last:
    SplitPoint = Partition(Array, First, Last)
    #recursively quicksort left half
    Quicksort(Array, First, SplitPoint - 1)
    #recursively quicksort right half
    Quicksort(Array, SplitPoint + 1, Last)
  return Array #return once fully sorted where index term >= last index

### Mergesort

Divide and conquer: split into a binary tree, then sort by merging.

**Divide:** take the list and split it in half until you get single-object arrays (which are considered sorted). From there, your sorted arrays are ready to conquer.

**Conquer:** merge two selected arrays by comparing their current values. Copy the smaller number into the main array and move that side's index forward; when one side runs out, copy the leftovers from the other. Tada — you have a merged (and sorted) array.

Since mergesort is recursive (usually), merging the binary tree happens as you exit the recursion calls, and you end up with 1 big merged and sorted array.

- Best/average/worst time: **O(n log n)**
- Extra space: **O(n)** for temporary arrays; not in-place in this implementation
- **Stable:** yes here, because equal values are taken from `left` first (`<=`)

In [16]:
def merge(array, low, mid, high):
  left = array[low : mid + 1]
  right = array[mid + 1 : high + 1]
  i = 0
  j = 0
  k = low
  while i < len(left) and j < len(right):
    if left[i] <= right[j]:
      array[k] = left[i]
      i += 1
    else:
      array[k] = right[j]
      j += 1
    k += 1
  while i < len(left):
    array[k] = left[i]
    i += 1
    k += 1
  while j < len(right):
    array[k] = right[j]
    j += 1
    k += 1
  return array

def mergeSort(array, low, high):
  if low < high:
    mid = (low + high) // 2
    mergeSort(array, low, mid)
    mergeSort(array, mid + 1, high)
    merge(array, low, mid, high)
  return array

<a id="complexity-lookup"></a>
## Complexity lookup

| Algorithm | Best | Average | Worst | Extra space | Needs sorted input? |
|---|---:|---:|---:|---:|---|
| Linear search | O(1) | O(n) | O(n) | O(1) | No |
| Binary search (recursive) | O(1) | O(log n) | O(log n) | O(log n) stack | Yes |
| Hash-table search | O(1) avg. | O(1) avg. | O(n) | O(1) per search | No |
| Insertion sort | O(n) | O(n²) | O(n²) | O(1) | — |
| Bubble sort (early exit) | O(n) | O(n²) | O(n²) | O(1) | — |
| Quicksort | O(n log n) | O(n log n) | O(n²) | O(log n) avg. stack | — |
| Mergesort | O(n log n) | O(n log n) | O(n log n) | O(n) | — |

### Exam distinctions
- **Search result:** return an index when found and a sentinel such as `-1` when absent.
- **In-place:** mutates the given list using O(1) extra array storage.
- **Stable:** equal-key records keep their original relative order.
- **Divide and conquer:** divide into smaller subproblems, solve recursively, combine.

<a id="tests"></a>
## Test scaffold
Run all definition cells above first. Copies (`test_list[:]`) stop one in-place sort affecting the next test.

In [18]:
# scaffold — run cells above first, then try each algorithm yourself
reset_test_list()
print("linear:", LinearSearch(test_list, test_list[0]))

sorted_for_binary = sorted(test_list[:])  # binary search needs sorted input
print("binary on", sorted_for_binary, "->", BinarySearch(sorted_for_binary, test_list[0], 0, len(sorted_for_binary) - 1))

a = test_list[:]
print("insertion:", InsertionSort(a))

b = test_list[:]
print("bubble:", BubbleSort(b))

c = test_list[:]
print("quick:", Quicksort(c, 0, len(c) - 1))

d = test_list[:]
print("merge:", mergeSort(d, 0, len(d) - 1))

test list: [7, 9, 10, 6, 7, 4, 0, 1, 1, 8]
linear: 0
binary on [0, 1, 1, 4, 6, 7, 7, 8, 9, 10] -> 5
insertion: [0, 1, 1, 4, 6, 7, 7, 8, 9, 10]
bubble: [0, 1, 1, 4, 6, 7, 7, 8, 9, 10]
quick: [0, 1, 1, 4, 6, 7, 7, 8, 9, 10]
merge: [0, 1, 1, 4, 6, 7, 7, 8, 9, 10]
